# Natural language processing and large language models

*Data Science and AI, Module 8. Slides: `Module_8.pdf`.*

Keep the slides open on the right. Every heading below carries the slide
numbers it goes with, so the two stay in step.

Natural language processing (NLP) is about getting a computer to do useful
work with text people wrote for other people. By the end of this notebook
you will have taken text apart with patterns, pulled it off a web page,
cleaned it, scored its mood, trained a model to sort it, and looked inside
the dice roll that makes a large language model (LLM) write.

**How each section works.** A question comes first. Make a guess out loud or
in the chat before the cell runs. Then run it and compare.

**Four kinds of text.**

- Small **made-up** examples, always labelled as made up.
- The **Titanic passenger list** from the ML challenge (`titanic_train.csv`),
  because the names hide something useful.
- **2,000 film reviews**, 1,000 positive and 1,000 negative, from Bo Pang and
  Lillian Lee, "A Sentimental Education", Proceedings of the ACL, 2004
  (polarity dataset v2.0, distributed with NLTK with the authors'
  permission).
- One live web page, `books.toscrape.com`, a site built for scraping
  practice. Its prices are random and mean nothing.

The notebook is in six parts, one per section of the deck. Each part loads
what it needs, so you can start at any part.

## Before the first session: one-off downloads

The `dsai` environment already has NLTK and spaCy. Both need some data
files the first time. Run the cell below once; afterwards it takes a second.

spaCy's English model is installed from a terminal, not from Python. If the
cell says it is missing, open the Anaconda Prompt (Windows) or Terminal
(Mac), run `conda activate dsai`, then the command the cell prints.

In [1]:
import nltk
import spacy

# Each download is skipped quickly if it is already on this computer.
nltk.download("vader_lexicon", quiet=True)   # word list for sentiment (Part 4)
nltk.download("stopwords", quiet=True)       # common words (Part 3)
nltk.download("wordnet", quiet=True)         # dictionary for lemmas (Part 3)
nltk.download("movie_reviews", quiet=True)   # the 2,000 film reviews

print("nltk", nltk.__version__)
print("spacy", spacy.__version__)

if spacy.util.is_package("en_core_web_sm"):
    print("spaCy English model: installed")
else:
    print("spaCy English model missing. Run this once in a terminal:")
    print("    python -m spacy download en_core_web_sm")    

nltk 3.10.2
spacy 3.8.14
spaCy English model: installed


# Part 4: slides 38-48

Sentiment analysis, first with a word list and rules, then measured on
2,000 real reviews.

In [2]:
import numpy as np
import pandas as pd
from nltk.corpus import movie_reviews
from nltk.sentiment import SentimentIntensityAnalyzer

scorer = SentimentIntensityAnalyzer()

## Slides 39-42: what sentiment analysis measures

Sentiment analysis pulls opinions out of text. The simplest version is
**polarity**: positive, negative or neutral. The variations on slides 41-42
(emotions, aspects such as "battery life", intent such as "complaint") are
the same idea with more labels.

**VADER** is a rule-based scorer that ships with NLTK (Hutto and Gilbert,
2014). It has a list of about 7,500 words and emoticons, each rated by
people from very negative to very positive, plus rules for "not", "very",
capitals and exclamation marks. Its `compound` score runs from -1 (most
negative) to +1 (most positive). Its authors suggest calling a text positive
at 0.05 or above and negative at -0.05 or below; in between is neutral.

The sentences below are the examples from slides 41-46. Before I run this:
pick two that you think VADER gets wrong.

In [3]:
examples = [
    "Customer support is killing me",
    "You are killing it",
    "Yeah. Sure.",
    "This product is second to none.",
    "The package is nice.",
    "The package is red.",
    "The battery life of this phone is too short.",
    ":D",
    "👍",
]

rows = []
for text in examples:
    rows.append({"text": text, "compound": scorer.polarity_scores(text)["compound"]})

print(pd.DataFrame(rows))

                                           text  compound
0                Customer support is killing me   -0.4019
1                            You are killing it   -0.6597
2                                   Yeah. Sure.    0.5423
3               This product is second to none.    0.0000
4                          The package is nice.    0.4215
5                           The package is red.    0.0000
6  The battery life of this phone is too short.    0.0000
7                                            :D    0.5106
8                                             👍    0.0000


Things to notice:

- "You are killing it" (praise) scores **negative**: the word list sees
  "killing", not the idiom;
- "Yeah. Sure." (sarcasm, in answer to "nice customer experience?") scores
  positive;
- "second to none" and "battery life ... too short" score 0: no single word
  on the list carries the opinion;
- "nice" scores positive and "red" scores 0, the subjective/objective
  difference from slide 44;
- the emoticon `:D` is on VADER's list; NLTK's copy has no entry for the
  👍 emoji (the separate `vaderSentiment` package translates emoji into
  words first).

Now its rules. Before I run this: what does one "not" do?

In [4]:
for text in ["I love it", "I do not love it", "I LOVE it!!!"]:
    print(f"{text:<18} {scorer.polarity_scores(text)['compound']}")

I love it          0.6369
I do not love it   -0.5216
I LOVE it!!!       0.7788


The "not" rule flips the score, and capitals plus exclamation marks push it
further. Hand-written rules like these are cheap, need no training data and
can be explained line by line. The question is how well they do on real
text.

Your turn: score a customer review

Read the review below. Before running any code, label each line as positive, negative or neutral.
Run the cell to get VADER’s compound score for each line.
Where does VADER disagree with you? Why might that be?
Score the whole review at once. Do you agree with the result?

Rule of thumb: compound ≥ 0.05 is positive, ≤ -0.05 is negative, anything in between is neutral.

In [5]:
# A short customer review, one sentence per line
review = """I ordered the headphones on Monday and they arrived on Friday.
The sound quality is amazing!
The ear cushions are not comfortable at all.
Battery life is okay, nothing special.
Customer service never replied to my email.
Would I buy them again? Absolutely not."""

# Split the review into a list of lines
lines = review.split("\n")

# Empty list to collect results
rows = []

# Score each line
for line in lines:

    # Get the compound score for this line
    compound = scorer.polarity_scores(line)["compound"]

    # Turn the score into a label
    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    # Store the line, score and label
    rows.append({"text": line, "compound": compound, "label": label})

# Show the results as a table
results = pd.DataFrame(rows)
print(results)

# Now score the whole review in one go
overall = scorer.polarity_scores(review)["compound"]
print("Whole review:", overall)

                                                text  compound     label
0  I ordered the headphones on Monday and they ar...    0.0000   neutral
1                      The sound quality is amazing!    0.6239  positive
2       The ear cushions are not comfortable at all.   -0.4023  negative
3             Battery life is okay, nothing special.   -0.0920  negative
4        Customer service never replied to my email.    0.0000   neutral
5            Would I buy them again? Absolutely not.    0.0000   neutral
Whole review: 0.2575


In [6]:
scorer.polarity_scores("The sound quality is amazing!") #["compound"]

{'neg': 0.0, 'neu': 0.494, 'pos': 0.506, 'compound': 0.6239}

### How VADER computes the compound score

**Example:** "The sound quality is amazing!"

1. **Look up each word** in VADER's word list (words not on the list score 0):

   | Word | Score |
   |---|---|
   | the, sound, quality, is | 0 |
   | amazing | +2.8 |

2. **Apply the rules.** Each `!` adds 0.292. (Other rules: "not" flips a score, "very" boosts it, CAPS boosts it.)

3. **Add up:** $x = 2.8 + 0.292 = 3.092$

4. **Squash into the range -1 to +1:**

$$\text{compound} = \frac{x}{\sqrt{x^2 + 15}} = \frac{3.092}{\sqrt{3.092^2 + 15}} = 0.6239$$

- $x$ = the raw total from step 3
- $15$ = a fixed number chosen by VADER's authors; it stops one strong word from pushing the score to the maximum
- The result never quite reaches ±1

**Rule of thumb:** compound ≥ 0.05 is positive, ≤ -0.05 is negative, anything in between is neutral.

In [7]:
import math

# Raw total: "amazing" (2.8) plus one "!" (0.292)
x = 2.8 + 0.292

# Squash into -1 to +1
compound = x / math.sqrt(x * x + 15)

# Compare our answer with VADER's
print("By hand:", round(compound, 4))
print("VADER:  ", scorer.polarity_scores("The sound quality is amazing!")["compound"])

By hand: 0.6239
VADER:   0.6239


In [8]:
scorer.lexicon['killing']

-3.4

In [9]:
#
import spacy

# Load spaCy's English model (this creates nlp)
nlp = spacy.load("en_core_web_sm")

# Run the sentence through spaCy
doc = nlp("The sound quality is amazing!")

# spaCy has no sentiment scorer, so this stays 0.0
print(doc.sentiment)

0.0

## VADER on 2,000 real reviews

Each review was labelled positive or negative from the rating its author
gave, extracted automatically by the dataset's makers. VADER never saw the labels. The text has been lowercased and
the punctuation spaced out by the dataset's makers. Since every review here
is positive or negative, call a review positive when `compound` is 0 or
above.

Before I run this: a coin toss gets 50%. What accuracy do you expect from
VADER?

In [10]:
texts = []
labels = []
for label in ["neg", "pos"]:
    for file_id in movie_reviews.fileids(label):
        texts.append(movie_reviews.raw(file_id))
        labels.append(label)

reviews = pd.DataFrame({"text": texts, "label": labels})

compound_scores = []
for text in reviews["text"]:
    compound_scores.append(scorer.polarity_scores(text)["compound"])
reviews["compound"] = compound_scores
reviews["vader"] = np.where(reviews["compound"] >= 0, "pos", "neg")

accuracy = (reviews["vader"] == reviews["label"]).mean()
print("VADER accuracy:", round(accuracy, 3))

VADER accuracy: 0.635


Better than a coin toss, but not by as much as you might hope. A crosstab
from Module 5 shows which way it goes wrong: true label down the side,
VADER's call across the top.

In [11]:
print(pd.crosstab(reviews["label"], reviews["vader"]))

vader  neg  pos
label          
neg    444  556
pos    174  826


VADER is far better at spotting positive reviews than negative ones; more
than half the negative reviews are called positive. A likely reason, worth checking by reading a few: critics often retell the
plot and praise parts of a film before saying it fails, so the good words
can outnumber the bad.

VADER's own documentation describes it as "specifically attuned to
sentiments expressed in social media", and these are long essays. The lesson is general: a tool can be good
and still be wrong for your text. Measure it on labelled examples of
**your** text before trusting it.


# Part 5: Text classification

**Goal:** build a model that reads a movie review and predicts whether it is positive or negative.

The same supervised steps as every model in the course:

1. Load the data
2. Split into train and test
3. Turn text into numbers
4. Train a model
5. Check how good it is
6. Use it on new text

The only new step is 3: a model needs numbers, and a review is text.

In [12]:
# Tools for tables
import pandas as pd

# The 2,000 movie reviews that ship with NLTK
from nltk.corpus import movie_reviews

# Ask the dataset for its labels
print(movie_reviews.categories())   # ['neg', 'pos']

# Empty lists to fill
texts = []
labels = []

# Go through the negative reviews, then the positive ones
for label in ["neg", "pos"]:

    # Each review is stored in its own file
    for file_id in movie_reviews.fileids(label):

        # Read the review text
        text = movie_reviews.raw(file_id)

        # Keep the text and its label
        texts.append(text)
        labels.append(label)

# Put everything into one table
reviews = pd.DataFrame({"text": texts, "label": labels})

# How many of each label?
print(reviews["label"].value_counts())
print()

# Peek at the start of the first review
print(reviews["text"][0][:150])
display(reviews.head(1))

['neg', 'pos']


label
neg    1000
pos    1000
Name: count, dtype: int64

plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continue


,text,label
0,"plot : two teen couples go to a church party ,...",neg


## Step 2: Split into train and test

- **X** is the input (the review text)
- **y** is what we want to predict (the label)

75% of the reviews are for training and 25% are kept back for testing.
`stratify=y` keeps the same pos/neg mix in both parts.

In [13]:
from sklearn.model_selection import train_test_split

# X = the review text (the input)
X = reviews["text"]

# y = the label we want to predict
y = reviews["label"]

# 75% to train, 25% to test, same pos/neg mix in both
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

print("training reviews:", len(X_train))
print("test reviews:    ", len(X_test))

training reviews: 1500
test reviews:     500


## Step 3: Turn text into numbers (TF-IDF)

**TF-IDF** gives every word a score in every review:

- **high** if the word appears often in *this* review
- **low** if the word appears in *lots of* reviews (like "the")

The result is a table: one row per review, one column per word.

Let's see it on three tiny made-up reviews first.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Made up: three tiny "reviews"
little_reviews = [
    "the great film",
    "the boring film",
    "the great great story",
]

# Create the vectoriser
little_vectoriser = TfidfVectorizer()

# Learn the words, then turn each sentence into numbers
little_numbers = little_vectoriser.fit_transform(little_reviews)

# Show as a table: one row per sentence, one column per word
little_table = pd.DataFrame(
    little_numbers.toarray().round(2),
    columns=little_vectoriser.get_feature_names_out(),
)
print(little_table)

   boring  film  great  story   the
0    0.00  0.62   0.62   0.00  0.48
1    0.72  0.55   0.00   0.00  0.43
2    0.00  0.00   0.79   0.52  0.31


What to notice:

- "the" is in every review, so it gets the **lowest** score in each row.
- "boring" appears in only one review, so it gets the **highest** score in that row.
- A 0 means the word is not in that review.

Now the real reviews. One rule:
**learn the words from the training reviews only** (`fit_transform`),
then convert the test reviews with that same word list (`transform`).
Same rule as scaling in earlier modules: fit on train, apply to test.

In [15]:
# Create the vectoriser
vectoriser = TfidfVectorizer()

# Learn the vocabulary from the TRAINING reviews only, then convert them
X_train_numbers = vectoriser.fit_transform(X_train)

# Convert the test reviews using that same vocabulary (no learning)
X_test_numbers = vectoriser.transform(X_test)

# Rows = reviews, columns = words
print("training table:", X_train_numbers.shape)
print("test table:    ", X_test_numbers.shape)
#display(pd.DataFrame(X_train_numbers.toarray().round(2), columns=vectoriser.get_feature_names_out()))

training table: (1500, 35283)
test table:     (500, 35283)


## Step 4: Train a model

Logistic regression, the same classifier as Module 5.
It learns which words push a review towards "pos" and which towards "neg".

In [16]:
from sklearn.linear_model import LogisticRegression

# Create the model
model = LogisticRegression(max_iter=1000)

# Learn which words go with "pos" and which with "neg"
model.fit(X_train_numbers, y_train)

print("model trained")

model trained


## Step 5: How good is it?

- **Accuracy** is the share of test reviews predicted correctly.
- The **confusion table** shows where the mistakes are.

In [17]:
# Predict a label for every test review
predictions = model.predict(X_test_numbers)

# Accuracy = share of test reviews predicted correctly
accuracy = (predictions == y_test).mean()
print("accuracy:", round(accuracy, 3))
print()

# Confusion table: true label (rows) vs predicted label (columns)
table = pd.crosstab(y_test, predictions, rownames=["true"], colnames=["predicted"])
print(table)

accuracy: 0.83

predicted  neg  pos
true               
neg        212   38
pos         47  203


In [18]:
# Count each of the four outcomes ("pos" is the positive class)
true_pos = ((y_test == "pos") & (predictions == "pos")).sum()
true_neg = ((y_test == "neg") & (predictions == "neg")).sum()
false_pos = ((y_test == "neg") & (predictions == "pos")).sum()
false_neg = ((y_test == "pos") & (predictions == "neg")).sum()

# Build the table with a label in every cell
labelled_table = pd.DataFrame(
    {
        "predicted neg": [f"{true_neg} (true negative)", f"{false_neg} (false negative)"],
        "predicted pos": [f"{false_pos} (false positive)", f"{true_pos} (true positive)"],
    },
    index=["true neg", "true pos"],
)
print(labelled_table)
print()

# Accuracy = correct predictions / all predictions
correct = true_pos + true_neg
total = true_pos + true_neg + false_pos + false_neg
print("accuracy:", correct, "/", total, "=", round(correct / total, 3))

                predicted neg        predicted pos
true neg  212 (true negative)  38 (false positive)
true pos  47 (false negative)  203 (true positive)

accuracy: 415 / 500 = 0.83


### Reading the confusion table

Rows are the **true** label. Columns are what the model **predicted**.
We call "pos" the positive class.

|  | predicted neg | predicted pos |
|---|---|---|
| **true neg** | 212 **true negatives**: negative reviews correctly called negative | 38 **false positives**: negative reviews wrongly called positive |
| **true pos** | 47 **false negatives**: positive reviews wrongly called negative | 203 **true positives**: positive reviews correctly called positive |

- The **diagonal** (212 and 203) holds the correct predictions.
- The **off-diagonal** (38 and 47) holds the mistakes.

$$\text{accuracy} = \frac{\text{correct}}{\text{total}} = \frac{212 + 203}{500} = \frac{415}{500} = 0.83$$

**What the mistakes tell us:** the model misses more positive reviews (47) than it wrongly
praises negative ones (38). It leans slightly towards calling a review negative.


## Step 6: Use it on new text

New text goes through the **same two steps**: the same vectoriser, then the same model.

`predict_proba` also gives the model's confidence: the chance the review is positive.

In [19]:
# Our own sentences
my_reviews = [
    "Brilliant acting and a great story.",
    "The worst film I have seen this year.",
    "The film was released in 1999.",
]

# Step 1: turn them into numbers with the SAME vectoriser
my_numbers = vectoriser.transform(my_reviews)

# Step 2: ask the model for a label
my_predictions = model.predict(my_numbers)

# Step 3: ask for the chances (column 0 = neg, column 1 = pos)
my_probabilities = model.predict_proba(my_numbers)

# Show each sentence with its label and its chance of being positive
for i in range(len(my_reviews)):
    review = my_reviews[i]
    label = my_predictions[i]
    chance_pos = my_probabilities[i][1]
    print(label, round(chance_pos, 4), "->", review) # 4

pos 0.6502 -> Brilliant acting and a great story.
neg 0.2263 -> The worst film I have seen this year.
neg 0.4991 -> The film was released in 1999.


The last sentence is a plain fact, and the model gives it a 50/50 chance: it has no idea.
It must still pick a label, so it says "neg". Always look at the confidence, not just the label.

## Which model is best?

Logistic regression is not the only classifier. Let's train five models on the
**same** training numbers and test them on the **same** test numbers.

| Model | Seen in |
|---|---|
| naive Bayes | Module 5 |
| logistic regression | Module 5 |
| linear SVM | new: finds the widest gap between the two classes |
| decision tree | Module 7 |
| random forest | Module 7 |

**Before you run it:** which model wins? Where does the single decision tree land?

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# The models to compare: a name and a fresh, untrained model
models = {
    "naive Bayes": MultinomialNB(),
    "logistic regression": LogisticRegression(max_iter=1000),
    "linear SVM": LinearSVC(),
    "decision tree": DecisionTreeClassifier(random_state=1),
    "random forest": RandomForestClassifier(n_estimators=100, random_state=1),
} # max_depth = 1,

# Empty list to collect one row per model
rows = []

for name, model in models.items():

    # Train on the training numbers from Step 3
    model.fit(X_train_numbers, y_train)

    # Accuracy on the test numbers
    accuracy = model.score(X_test_numbers, y_test)

    # Store the result
    rows.append({"model": name, "accuracy": round(accuracy, 3)})

# Show the results, best first
results = pd.DataFrame(rows)
results = results.sort_values("accuracy", ascending=False)
print(results.to_string(index=False))

              model  accuracy
         linear SVM     0.854
logistic regression     0.830
        naive Bayes     0.806
      random forest     0.774
      decision tree     0.680


- The **linear** models win: they weigh up all 35,000 words at once.
- The **single tree** is weakest: each split looks at one word, and one word at a time
  is a poor way to read a review.
- The **forest** (many trees voting) recovers most of the gap.

How does VADER from Part 4, which learns nothing, compare on the same test reviews?

In [21]:
from nltk.sentiment import SentimentIntensityAnalyzer

# VADER from Part 4: a word list, no training at all
scorer = SentimentIntensityAnalyzer()

# Label each test review with VADER
vader_labels = []
for text in X_test:

    # Compound score for the whole review
    compound = scorer.polarity_scores(text)["compound"]

    # 0 or above counts as positive
    if compound >= 0:
        vader_labels.append("pos")
    else:
        vader_labels.append("neg")

# Accuracy = share that match the true labels
vader_accuracy = (pd.Series(vader_labels, index=y_test.index) == y_test).mean()
print("VADER accuracy:", round(vader_accuracy, 3))

VADER accuracy: 0.646


Learning from labelled examples wins.

## One function for any dataset

The steps are always the same: split, turn text into numbers, train, test.
Let's put them in one function so we can try other kinds of text.

In [22]:
def try_models(texts, labels):

    # Step 1: split, keeping the same label mix in train and test
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.25, random_state=1, stratify=labels
    )

    # Step 2: learn the words from the training texts, convert both sets
    vectoriser = TfidfVectorizer()
    X_train_numbers = vectoriser.fit_transform(X_train)
    X_test_numbers = vectoriser.transform(X_test)

    # Step 3: train and test each model
    models = {
        "naive Bayes": MultinomialNB(),
        "logistic regression": LogisticRegression(max_iter=1000),
        "linear SVM": LinearSVC(),
        "decision tree": DecisionTreeClassifier(random_state=1),
        "random forest": RandomForestClassifier(n_estimators=100, random_state=1),
    }

    rows = []
    for name, model in models.items():
        model.fit(X_train_numbers, y_train)
        accuracy = model.score(X_test_numbers, y_test)
        rows.append({"model": name, "accuracy": round(accuracy, 3)})

    # Step 4: show the results, best first
    results = pd.DataFrame(rows)
    results = results.sort_values("accuracy", ascending=False)
    print(results.to_string(index=False))


# Check: the movie reviews should give the same numbers as before
try_models(reviews["text"], reviews["label"])

              model  accuracy
         linear SVM     0.854
logistic regression     0.830
        naive Bayes     0.806
      random forest     0.774
      decision tree     0.680


## Tweets (social media) data

5,000 positive and 5,000 negative tweets that ship with NLTK.
Much **shorter** than movie reviews.

In [23]:
import nltk
from nltk.corpus import twitter_samples

# One-off download (about 7 MB)
nltk.download("twitter_samples", quiet=True)

# Two lists of tweets
positive_tweets = twitter_samples.strings("positive_tweets.json")
negative_tweets = twitter_samples.strings("negative_tweets.json")

# Build one table: text + label
tweet_texts = positive_tweets + negative_tweets
tweet_labels = ["pos"] * len(positive_tweets) + ["neg"] * len(negative_tweets)
tweets = pd.DataFrame({"text": tweet_texts, "label": tweet_labels})

print(tweets["label"].value_counts())
print()
print(positive_tweets[1])
print("@@@@@@@@@@@@@@@@@@@@@@@@@")
print(negative_tweets[1])

label
pos    5000
neg    5000
Name: count, dtype: int64

@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!
@@@@@@@@@@@@@@@@@@@@@@@@@
Everything in the kids section of IKEA is so cute. Shame I'm nearly 19 in 2 months :(


In [24]:
try_models(tweets["text"], tweets["label"])

              model  accuracy
logistic regression     0.767
         linear SVM     0.763
        naive Bayes     0.759
      random forest     0.755
      decision tree     0.691


Lower than movie reviews: a tweet is a handful of words, so there is less to learn from.

Notice the tweets end in `:)` or `:(`. **Before you run it:** how well would a
one-line rule do, "if the tweet has `:)`, call it positive"?

In [25]:
# A one-line rule: does the tweet contain a smiley?
rule_labels = []
for text in tweets["text"]:
    if ":)" in text:
        rule_labels.append("pos")
    else:
        rule_labels.append("neg")

# How often does the rule match the true label?
rule_accuracy = (pd.Series(rule_labels) == tweets["label"]).mean()
print("smiley rule accuracy:", round(rule_accuracy, 3))

smiley rule accuracy: 0.854
